# Анализ рекламных событий

### Цель проекта
Определить причину резкого роста количества взаимодействий с рекламными объявлениями в один из дней.

### Задачи проекта
- рассчитать ежедневную статистику по рекламным объявлениям: кол-во всех событий по дням, показов, кликов, уникальных объявлений и кампаний;
- определить день с аномальным ростом количества взаимодействий и выяснить причину скачка;
- рассчитать CTR объявлений и проанализировать распределение показателя;
- проверить качество логирования событий: найти объявления с кликами без показов и случаи, когда клик произошёл раньше показа, определить платформы;
- сравнить CTR с видео и без видео;
- рассчитать рекламный доход по дням с учётом моделей оплаты CPC и CPM, выявить дни с максимальным и минимальным доходом;
- выявить популярную платформу для размещения рекламных объявлений.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

#### 1. Загрузка и первичный осмотр данных

In [2]:
ads_data = pd.read_csv('data/ads_data.csv')

In [3]:
ads_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3176714 entries, 0 to 3176713
Data columns (total 12 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   ad_id                  int64  
 1   time                   int64  
 2   event                  object 
 3   date                   object 
 4   ad_cost_type           object 
 5   has_video              int64  
 6   client_union_id        int64  
 7   campaign_union_id      int64  
 8   platform               object 
 9   ad_cost                float64
 10  target_audience_count  int64  
 11  user_id                int64  
dtypes: float64(1), int64(7), object(4)
memory usage: 290.8+ MB


In [4]:
ads_data.isna().sum()

ad_id                    0
time                     0
event                    0
date                     0
ad_cost_type             0
has_video                0
client_union_id          0
campaign_union_id        0
platform                 0
ad_cost                  0
target_audience_count    0
user_id                  0
dtype: int64

#### 2. Подготовка данных

In [5]:
#Преобразуем столбцы с датой и временем в формат datetime
ads_data['date'] = pd.to_datetime(ads_data.date)
ads_data['full_date'] = pd.to_datetime(ads_data.time, unit='s')

#### 3. Расчет ежедневной статистики по рекламным объявлениям

In [6]:
# Считаем количество кликов и показов по дням
ads_data_stat_of_day = (ads_data.groupby(['date','event'], as_index = False)
                        .agg({'ad_id': 'count'})
                        .pivot(index= 'date', columns='event', values='ad_id')
                        .reset_index())
ads_data_stat_of_day

event,date,click,view
0,2019-04-01,735,34832
1,2019-04-02,1518,145811
2,2019-04-03,1678,214851
3,2019-04-04,1517,126182
4,2019-04-05,501595,1783288
5,2019-04-06,80420,284287


In [7]:
# Считаем количество уникальных объявлений по дням
nunique_ad_id = (ads_data.groupby('date', as_index = False)
                        .agg(nunique_ad_id = ('ad_id', 'nunique')))

In [8]:
# Считаем количество уникальных рекламных кампаний по дням
nunique_campaign_id = (ads_data.groupby('date', as_index = False)
                        .agg(nunique_campain_id = ('campaign_union_id', 'nunique')))

In [9]:
# Объединяем ежедневную статистику в одну таблицу
ads_data_stat_of_day = (ads_data_stat_of_day.merge(nunique_ad_id, on= 'date')
                        .merge(nunique_campaign_id, on= 'date'))

In [10]:
ads_data_stat_of_day

,date,click,view,nunique_ad_id,nunique_campain_id
0,2019-04-01,735,34832,49,49
1,2019-04-02,1518,145811,146,146
2,2019-04-03,1678,214851,179,177
3,2019-04-04,1517,126182,150,147
4,2019-04-05,501595,1783288,131,130
5,2019-04-06,80420,284287,61,60


**Выводы:** по ежедневной статистике видно, что 2019-04-05 количество событий резко выросло. При этом количество уникальных объявлений и рекламных кампаний не изменилось. Это означает, что аномальный рост связан скорее всего с резким ростом активности по уже существующим объявлениям.

#### 4. Выяснение причины скачка

In [11]:
# Кол-во событий по объявлениям за аномальный день
ads_data[ads_data.date == '2019-04-05'] \
    .groupby('ad_id') \
    .agg({'time': 'count'}) \
    .sort_values('time', ascending=False) \
    .head()

,time
ad_id,
112583,2166611
29927,27186
44635,8268
46583,7327
44956,5656


In [12]:
# Объявление 112583 имеет максимальное количество событий в аномальный день. Проверим, за счёт каких событий произошёл рост
ads_data.query('ad_id == 112583').groupby('event').agg({'ad_id': 'count'})

,ad_id
event,
click,580436
view,1934788


**Выводы:** Скачок 2019-04-05 связан преимущественно с активностью по одному рекламному объявлению. При этом он проявляется не только в росте показов, но и в увеличении количества кликов. Можно предположить, что это связано с выгрузкой новой крупной рекламы.

#### 5. Анализ CTR объявлений

CTR - показатель кликабельности:
CTR = количество кликов / количество показов


In [14]:
# Создадим сводную таблицу по id и event, где будет удобнее посчитать и отразить CTR
ads_data_by_ad =  (ads_data.groupby(['ad_id','event'], as_index = False)
                        .agg({'time': 'count'})
                        .pivot(index= 'ad_id', columns='event', values='time')
                        .reset_index())
ads_data_by_ad

event,ad_id,click,view
0,3,9.0,490.0
1,2132,1.0,95.0
2,2276,2.0,1454.0
3,2475,NaN,132.0
4,2643,3.0,286.0
...,...,...,...
350,121941,1.0,640.0
351,121943,15.0,1722.0
352,122042,1.0,155.0
353,122063,1.0,260.0


In [15]:
# Рассчитаем CTR и выведем топ-10 по этому критерию
ads_data_by_ad['ctr'] = ads_data_by_ad.click / ads_data_by_ad.view
ads_data_by_ad.sort_values('ctr', ascending = False).head(10)

event,ad_id,click,view,ctr
289,112583,580436.0,1934788.0,0.300000
324,119450,258.0,1254.0,0.205742
125,38575,43.0,257.0,0.167315
144,40968,29.0,217.0,0.133641
207,45642,42.0,344.0,0.122093
283,110924,11.0,95.0,0.115789
194,45043,28.0,245.0,0.114286
96,35034,112.0,997.0,0.112337
5,4585,53.0,476.0,0.111345
328,120347,236.0,2168.0,0.108856


In [16]:
# Медианное значение CTR
ads_data_by_ad.ctr.median()

0.010753240746688594

In [17]:
# Среднее значение CTR
ads_data_by_ad.ctr.mean()

np.float64(0.020628096080757954)

**Выводы:** CTR объявлений распределён неравномерно: среднее значение выше медианного, значит, отдельные объявления с высокими показателями CTR смещают среднее вверх.

Топ-10 объявлений по CTR показывает, какие объявления имеют наибольшую долю кликов относительно показов. При этом высокий CTR нужно интерпретировать осторожно: если объявление получило небольшое количество показов, показатель может быть нестабильным.


#### 6. Проверка качества логирования событий

In [18]:
# Посчитаем кол-во событий по платформам и id
ads_data_by_ad_platform = (ads_data.groupby(['ad_id', 'event', 'platform'], as_index                            =False)
                           .agg(count_event = ('event', 'count'))
                           .pivot(index = ['ad_id', 'platform'], columns = 'event', values = 'count_event')
                           .reset_index()
                           .fillna(0)
                           .sort_values(['view', 'ad_id']))

In [19]:
# Оставляем объявления, у которых есть клики, но нет показов
ads_data_bag = ads_data_by_ad_platform.query('view == 0 and click > 0')
ads_data_bag

event,ad_id,platform,click,view
150,25665,android,6.0,0.0
151,25665,ios,4.0,0.0
152,25665,web,4.0,0.0
231,30381,android,39.0,0.0
232,30381,ios,25.0,0.0
233,30381,web,13.0,0.0
450,41424,android,1.0,0.0
451,41424,ios,1.0,0.0
482,42241,android,60.0,0.0
483,42241,ios,46.0,0.0


In [20]:
# Считаем количество проблемных объявлений по платформам
count_ad_bug_by_platform = ads_data_bag.groupby('platform').agg(count_bag_ad = ('ad_id', 'nunique'))
count_ad_bug_by_platform

,count_bag_ad
platform,
android,9
ios,9
web,8


In [21]:
# Считаем общее количество уникальных объявлений по платформам
total_by_platform = ads_data.groupby('platform', as_index = False).agg(count_ad = ('ad_id', 'nunique'))

In [22]:
# Объединяем таблицы и считаем долю проблемных объявлений
ads_data_bag = ads_data_bag.merge(count_ad_bug_by_platform, on = 'platform').merge(total_by_platform, on = 'platform')

ads_data_bag['bug_share'] = (ads_data_bag['count_bag_ad'] / ads_data_bag['count_ad']).round(4)

ads_data_bag.groupby('platform').agg({'bug_share': 'max'})

,bug_share
platform,
android,0.0254
ios,0.0254
web,0.0226


**Выводы:** проблема с объявлениями, у которых есть клики, но нет показов, встречается на всех платформах. Для бизнесовой интерпретации я рассчитала их долю среди всех объявлений платформы. Эта доля показывает, на какой платформе проблема выражена сильнее относительно общего объёма объявлений.

Такую платформу стоит проверить в первую очередь: возможно, ошибка связана с особенностями логирования, загрузки или обработки событий на конкретной платформе.

#### 7. Сравнение CTR с видео и без видео

In [23]:
# Вернемся к df, в котором рассчитан ctr. Заменим NaN значения на 0, для удобства анализа
ads_data_by_ad= ads_data_by_ad.fillna(0)

In [24]:
# Определяем, есть ли видео у каждого объявления
has_video_ad_id = ads_data.groupby('ad_id').agg({'has_video': 'max'})

In [25]:
# Добавляем признак наличия видео к таблице с CTR по объявлениям
ads_data_by_ad = ads_data_by_ad.merge(has_video_ad_id, on ='ad_id')

In [26]:
# Сравниваем CTR объявлений с видео и без видео
stat_ctr_by_has_video = ads_data_by_ad.groupby('has_video').agg({'ctr': 'describe'})
stat_ctr_by_has_video

ctr                                                         \
           count      mean       std  min       25%       50%       75%   
has_video                                                                 
0          349.0  0.018026  0.029248  0.0  0.002933  0.009215  0.021429   
1            6.0  0.003539  0.005620  0.0  0.000425  0.001709  0.002719   

                    
               max  
has_video           
0          0.30000  
1          0.01476

**Вывод:** По рассчитанным статистикам видно, что объявления без видео в текущих данных имеют более высокий средний и медианный CTR, чем объявления с видео. Однако в данных всего 6 объявлений с видео, поэтому выборка слишком маленькая для уверенного обобщения.

#### 8. Расчет дохода по дням

Рассчитаем рекламный доход по дням с учётом модели оплаты:
- для объявлений с типом оплаты CPC доход начисляется за клики;
- для объявлений с типом оплаты CPM доход начисляется за показы, при этом стоимость одного показа равна ad_cost / 1000.

In [27]:
# Суммарный доход по CPC объявлениям за весь период
sum_ad_cost_of_CPC = ads_data[(ads_data['ad_cost_type'] == 'CPC') & (ads_data['event'] == 'click')].ad_cost.sum()

In [28]:
# Суммарный доход по CPM объявлениям за весь период
sum_ad_cost_of_CPM = (ads_data[(ads_data['ad_cost_type'] == 'CPM') & (ads_data['event'] == 'view')].ad_cost / 1000).sum()
sum_ad_cost_of_CPM

np.float64(497090.6777999998)

In [29]:
# Доход по CPC по дням
CPC_of_day = (ads_data.query('ad_cost_type == "CPC" & event == "click"')
 .groupby('date',as_index = False)
 .agg(sum_cost_CPC = ('ad_cost', 'sum')))

In [30]:
# Доход по CPM по дням
CPM_of_day = (ads_data.query('ad_cost_type == "CPM" & event == "view"')
 .groupby('date', as_index = False)
 .agg(sum_cost_CPM = ('ad_cost', lambda x: x.sum() / 1000)))

In [31]:
# Объединяем доход по CPC и CPM
ad_cost_by_the_date =  CPC_of_day.merge(CPM_of_day, on = 'date')

In [32]:
# Считаем общий доход по дням
ad_cost_by_the_date['total_sum'] = ad_cost_by_the_date.sum_cost_CPC	+ ad_cost_by_the_date.sum_cost_CPM
ad_cost_by_the_date

,date,sum_cost_CPC,sum_cost_CPM,total_sum
0,2019-04-01,7036.9,6122.8123,13159.7123
1,2019-04-02,7663.7,26173.3051,33837.0051
2,2019-04-03,38597.2,34612.3170,73209.5170
3,2019-04-04,26878.0,19416.3568,46294.3568
4,2019-04-05,4381.2,354178.5490,358559.7490
5,2019-04-06,253.0,56587.3376,56840.3376


In [33]:
# День с максимальным доходом
day_of_the_max_cost = ad_cost_by_the_date.loc[ad_cost_by_the_date['total_sum'].idxmax()]
day_of_the_max_cost

date            2019-04-05 00:00:00
sum_cost_CPC                 4381.2
sum_cost_CPM             354178.549
total_sum                358559.749
Name: 4, dtype: object

In [34]:
# День с минимальным доходом
day_of_the_min_cost = ad_cost_by_the_date.loc[ad_cost_by_the_date['total_sum'].idxmin()]
day_of_the_min_cost

date            2019-04-01 00:00:00
sum_cost_CPC                 7036.9
sum_cost_CPM              6122.8123
total_sum                13159.7123
Name: 0, dtype: object

**Выводы:** доход был рассчитан отдельно для двух моделей оплаты и совместно за день.
День с максимальным доходом совпадает с аномальным днём роста рекламных событий. Это ожидаемо, так как 2019-04-05 было зафиксировано резкое увеличение количества показов и кликов.
День с минимальным доходом показывает наименьший объём монетизируемых взаимодействий за период наблюдений.

#### 9. Выявим популярную платформу для размещения рекламных объявлений

In [35]:
# Возьмем df с кол-во уникальных объявлений на платформах, созданный раннее. И найдем платформу с максимальным кол-ом обьявлений
popular_platform = total_by_platform.loc[total_by_platform['count_ad'].idxmax()]
popular_platform

platform    android
count_ad        355
Name: 0, dtype: object

In [36]:
# Считаем количество показов по платформам
views_by_platform = (ads_data.query('event == "view"')
    .groupby('platform', as_index=False)
    .agg(view_count=('ad_id', 'count')))
views_by_platform

,platform,view_count
0,android,1295189
1,ios,776114
2,web,517948


In [37]:
# Считаем долю показов каждой платформы
views_by_platform['view_share'] = (views_by_platform['view_count'] / views_by_platform['view_count'].sum() * 100).round(2)

views_by_platform.sort_values('view_share', ascending=False)

,platform,view_count,view_share
0,android,1295189,50.02
1,ios,776114,29.97
2,web,517948,20.00


**Выводы:** Самая популярная платформа - android.
Судя по распределению долей показов по платформам на android объявления получают наибольший охват.

### Общий вывод:

В ходе анализа был найден день с резким ростом количества рекламных взаимодействий — 2019-04-05. В этот день значительно выросло количество как показов, так и кликов.

Основная причина скачка связана не с ростом числа рекламных объявлений или кампаний, а с активностью по одному объявлению — `112583`. На него пришлась большая часть событий в аномальный день, поэтому именно это объявление можно считать главным источником аномалии в данных.

Дополнительно был рассчитан CTR по объявлениям. Средний CTR оказался выше медианного, что говорит о неравномерном распределении показателя: отдельные объявления с высоким CTR смещают среднее значение вверх. Поэтому для оценки типичного уровня CTR лучше ориентироваться на медиану, а объявления с очень высоким CTR анализировать отдельно с учётом количества показов и кликов.

Также в данных были найдены признаки возможных ошибок логирования - выявлены случаи нарушения последовательности событий. Такие ситуации могут искажать рекламные метрики, поэтому их важно учитывать при дальнейшем анализе.

При сравнении объявлений с видео и без видео в текущих данных объявления без видео показали более высокий CTR. Однако этот вывод следует интерпретировать осторожно, так как обьявлений с видео в выборке значительно меньше.

Финансовый расчёт показал, что максимальный доход пришёлся на тот же день, в который наблюдался аномальный рост событий. Это логично, потому что увеличение количества показов и кликов напрямую влияет на доход по моделям оплаты CPM и CPC.

По платформам наибольшая доля показов приходится на android. Это показывает, что Android является основной платформой по объёму рекламных показов в рассматриваемых данных.

Таким образом, главная аномалия в данных связана с резким ростом активности по одному рекламному объявлению.